# 07 - Weather + push notification (ntfy.sh)

Combines `get_weather` with `send_notification`, which pushes straight to your phone via [ntfy.sh](https://ntfy.sh) instead of email or SMS - free, no signup, no account, and works from any country (no carrier/number involved at all).

**Setup:**
1. Install the **ntfy** app from the App Store (or Google Play) on your phone.
2. In the app, subscribe to a topic - pick a long, hard-to-guess name (e.g. `suresh-agent-alerts-38f2`). Anyone who knows this name can publish to it or read your notifications, since there's no login - treat it like a password, not a public label.
3. Enter that same topic name below when prompted.

## 1. Install dependencies

In [1]:
%pip install -q anthropic requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Set your credentials

The ntfy topic uses `getpass` too, since it functions as a password even though it's not a traditional API key.

In [1]:
import os
from getpass import getpass

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your ANTHROPIC_API_KEY: ")
if not os.environ.get("NTFY_TOPIC"):
    os.environ["NTFY_TOPIC"] = getpass("Enter your ntfy.sh topic name: ")

# Only needed if you hit: "anthropic-workspace-id is required when
# authenticating with an identity-linked API key" - leave blank to skip.
if not os.environ.get("ANTHROPIC_WORKSPACE_ID"):
    _workspace_id = input("ANTHROPIC_WORKSPACE_ID (leave blank if not needed): ").strip()
    if _workspace_id:
        os.environ["ANTHROPIC_WORKSPACE_ID"] = _workspace_id

## 3. Tools, the agent loop, and the `Agent` class

Same cost cap and turn-collapsing core as the other notebooks. The system prompt again explicitly limits `send_notification` to only fire when asked - same reasoning as `send_email` in notebooks 05/06.

In [3]:
import datetime
import json
import os

import anthropic

MODEL = "claude-haiku-4-5"
MAX_TOKENS = int(os.environ.get("AGENT_MAX_TOKENS", "1024"))

# claude-haiku-4-5 pricing, $/1M tokens - update if you switch models.
INPUT_COST_PER_MTOK = 1.00
OUTPUT_COST_PER_MTOK = 5.00

# Hard spending cap for this notebook kernel session.
MAX_COST_USD = float(os.environ.get("AGENT_MAX_COST_USD", "0.20"))


class BudgetExceededError(RuntimeError):
    pass
import requests

SYSTEM_PROMPT = (
    "You are a helpful assistant with get_weather and send_notification "
    "tools. Use get_weather for current conditions. Only call "
    "send_notification when the user explicitly asks you to notify, "
    "alert, or push something to their phone - never on your own "
    "initiative. Otherwise reply directly."
)

TOOLS = [
    {
        "name": "get_weather",
        "description": "Get current weather conditions for a location by city name.",
        "input_schema": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "City name, optionally with country, e.g. 'Paris, France'.",
                },
            },
            "required": ["location"],
        },
    },
    {
        "name": "send_notification",
        "description": "Push a notification to the user's phone via ntfy.sh.",
        "input_schema": {
            "type": "object",
            "properties": {
                "title": {"type": "string", "description": "Short notification title."},
                "message": {"type": "string", "description": "Notification body text."},
            },
            "required": ["title", "message"],
        },
    },
]

# WMO weather interpretation codes (used by Open-Meteo's weather_code field).
WMO_CODES = {
    0: "Clear sky", 1: "Mainly clear", 2: "Partly cloudy", 3: "Overcast",
    45: "Fog", 48: "Depositing rime fog",
    51: "Light drizzle", 53: "Moderate drizzle", 55: "Dense drizzle",
    56: "Light freezing drizzle", 57: "Dense freezing drizzle",
    61: "Slight rain", 63: "Moderate rain", 65: "Heavy rain",
    66: "Light freezing rain", 67: "Heavy freezing rain",
    71: "Slight snow fall", 73: "Moderate snow fall", 75: "Heavy snow fall",
    77: "Snow grains",
    80: "Slight rain showers", 81: "Moderate rain showers", 82: "Violent rain showers",
    85: "Slight snow showers", 86: "Heavy snow showers",
    95: "Thunderstorm", 96: "Thunderstorm with slight hail", 99: "Thunderstorm with heavy hail",
}


def _request_with_retry(method, url, attempts=2, **kwargs):
    last_exc = None
    for attempt in range(attempts):
        try:
            resp = requests.request(method, url, timeout=15, **kwargs)
            resp.raise_for_status()
            return resp
        except requests.RequestException as exc:
            last_exc = exc
    raise last_exc


def get_weather(location: str) -> str:
    """Free, no-API-key weather lookup via Open-Meteo."""
    try:
        geo_resp = _request_with_retry(
            "GET",
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": location, "count": 1},
        )
        geo_data = geo_resp.json()
    except requests.RequestException as exc:
        return f"Error: location lookup failed ({exc})"

    results = geo_data.get("results")
    if not results:
        return f"Error: could not find a location matching '{location}'."
    place = results[0]

    try:
        weather_resp = _request_with_retry(
            "GET",
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": place["latitude"],
                "longitude": place["longitude"],
                "current": "temperature_2m,wind_speed_10m,weather_code",
            },
        )
        weather_data = weather_resp.json()
    except requests.RequestException as exc:
        return f"Error: weather lookup failed ({exc})"

    current = weather_data.get("current", {})
    units = weather_data.get("current_units", {})
    condition = WMO_CODES.get(current.get("weather_code"), "Unknown conditions")
    place_label = place["name"] + ((", " + place["country"]) if place.get("country") else "")

    return (
        "Weather in " + place_label + ": " + condition + ", "
        + str(current.get("temperature_2m")) + units.get("temperature_2m", "\u00b0C")
        + ", wind " + str(current.get("wind_speed_10m")) + " " + units.get("wind_speed_10m", "km/h")
    )


def send_notification(title: str, message: str) -> str:
    """Push a notification via ntfy.sh - no account, no per-message cost.
    NTFY_TOPIC acts as a private "password"; anyone who knows it can also
    publish to (or read) it, so keep it unguessable and don't commit it."""
    topic = os.environ.get("NTFY_TOPIC")
    if not topic:
        return "Error: NTFY_TOPIC not set."
    try:
        _request_with_retry(
            "POST",
            f"https://ntfy.sh/{topic}",
            data=message.encode("utf-8"),
            headers={"Title": title},
        )
        return "Notification sent."
    except requests.RequestException as exc:
        return f"Error: failed to send notification ({exc})"


def execute_tool(name: str, tool_input: dict) -> str:
    if name == "get_weather":
        return get_weather(tool_input["location"])
    if name == "send_notification":
        return send_notification(tool_input["title"], tool_input["message"])
    return f"Error: unknown tool '{name}'"


MAX_PAUSE_RESUMES = 10


def build_client() -> anthropic.Anthropic:
    """Some API keys (personal keys not scoped to one workspace) require an
    anthropic-workspace-id header on every request - see
    https://platform.claude.com/docs/en/manage-claude/authentication#select-a-workspace.
    Set ANTHROPIC_WORKSPACE_ID if you hit: 'anthropic-workspace-id is
    required when authenticating with an identity-linked API key'."""
    workspace_id = os.environ.get("ANTHROPIC_WORKSPACE_ID")
    if workspace_id:
        return anthropic.Anthropic(
            default_headers={"anthropic-workspace-id": workspace_id}
        )
    return anthropic.Anthropic()


class Agent:
    """A minimal conversational agent that can call tools in a loop."""

    def __init__(self, client: anthropic.Anthropic | None = None):
        self.client = client or build_client()
        self.messages: list[dict] = []
        self.total_cost_usd = 0.0

    def send(self, user_input: str) -> str:
        turn_start = len(self.messages)
        self.messages.append({"role": "user", "content": user_input})

        resumes = 0
        while True:
            if self.total_cost_usd >= MAX_COST_USD:
                raise BudgetExceededError(
                    f"Session cost ${self.total_cost_usd:.4f} has reached the "
                    f"${MAX_COST_USD:.4f} cap (AGENT_MAX_COST_USD). Raise the "
                    "cap or start a new session to continue."
                )

            response = self.client.messages.create(
                model=MODEL,
                max_tokens=MAX_TOKENS,
                system=SYSTEM_PROMPT,
                tools=TOOLS,
                messages=self.messages,
            )
            self.total_cost_usd += (
                response.usage.input_tokens * INPUT_COST_PER_MTOK
                + response.usage.output_tokens * OUTPUT_COST_PER_MTOK
            ) / 1_000_000
            self.messages.append({"role": "assistant", "content": response.content})

            if response.stop_reason == "pause_turn":
                resumes += 1
                if resumes > MAX_PAUSE_RESUMES:
                    break
                continue

            if response.stop_reason != "tool_use":
                break

            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    result = execute_tool(block.name, block.input)
                    tool_results.append(
                        {
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": result,
                        }
                    )
            self.messages.append({"role": "user", "content": tool_results})

        reply = "".join(
            block.text for block in response.content if block.type == "text"
        )
        self.messages[turn_start:] = [
            {"role": "user", "content": user_input},
            {"role": "assistant", "content": reply},
        ]
        return reply


## 4. Create the agent

In [4]:
agent = Agent()
print(f"Agent ready (cap ${MAX_COST_USD:.4f} for this kernel session)")

Agent ready (cap $0.2000 for this kernel session)


## 5. Try it

Make sure the ntfy app is open/subscribed on your phone before running this - it should get a real push notification.

In [5]:
reply = agent.send(
    "What's the weather in Tokyo, and send a notification to my phone about it."
)
print(reply)
print(f"(session cost so far: ~${agent.total_cost_usd:.4f})")

Done! The weather in Tokyo is currently showing **light drizzle** with a temperature of **23.0°C** and wind at **2.4 km/h**. I've sent this information to your phone as a notification.
(session cost so far: ~$0.0036)


## 6. Optional: interactive chat loop

Type `exit` to stop.

In [ ]:
while True:
    user_input = input("You: ")
    if user_input.strip().lower() in {"exit", "quit"}:
        break
    try:
        reply = agent.send(user_input)
    except BudgetExceededError as exc:
        print(f"Agent: [stopped] {exc}")
        break
    print(f"Agent: {reply}  (session cost so far: ~${agent.total_cost_usd:.4f})")
